# UN data processing

Processes United Nations demographic series (e.g. age structure) into
per-country features.

In [ ]:
import os
import pandas as pd

# Read the TSV file
df = pd.read_csv('df.tsv', 
                sep='\t',  # Specify tab as separator
                encoding='utf-8',     # Try UTF-8 encoding
                )         

# Display basic information about the dataset
print("Dataset Shape:", df.shape)
print("\nFirst few rows:")
display(df.head())

print("\nColumn names and data types:")
display(df.dtypes)

print("\nBasic statistics:")
display(df.describe())

print("\nMissing values per column:")
display(df.isnull().sum())

In [ ]:
# strip the column name
# df.columns = df.columns.str.strip()

# Find all rows where % Stem Grads is 0.00
zero_stem_grads = df[df['% Stem Grads'] == 0.00]

# Display the count and some examples
print(f"Number of languages with 0.00% Stem Grads: {len(zero_stem_grads)}")
print("\nExample languages with 0.00% Stem Grads:")
print(zero_stem_grads[['primary_country', '% Stem Grads']].head(20))

In [ ]:
# find out all the unique countries with stem grads value is 0
unique_countries_with_0_stem_grads = df[df['% Stem Grads'] == 0]['primary_country'].unique()
print(unique_countries_with_0_stem_grads)

In [ ]:
# open UNESCO_UIS_GRAD_STEM.csv

unesco_uis_grad_stem = pd.read_csv('UNdata_Export_20250513_162817118.csv')

In [ ]:
# define a function to get the stem grads value from UNESCO_UIS_GRAD_STEM.csv. remember to find the most recent year
def get_stem_grads_value(country):
    all_data = unesco_uis_grad_stem[unesco_uis_grad_stem['Reference Area'] == country]
    # find the most recent year
    most_recent_year = all_data['Time Period'].max()
    # get the stem grads value
    stem_grads_value = all_data[all_data['Time Period'] == most_recent_year]['Observation Value'].values[0]
    return stem_grads_value

get_stem_grads_value('Australia')

In [ ]:
len(unique_countries_with_0_stem_grads)

In [ ]:
get_stem_grads_value('Afghanistan')

In [ ]:
stem_grads_value_dict = {}
# for each country in unique_countries_with_0_stem_grads, get the stem grads value
for country in unique_countries_with_0_stem_grads:
    country = country.strip()
    # try to get the stem grads value
    try:
        stem_grads_value = get_stem_grads_value(country)
        print(country, stem_grads_value)
        # save the stem grads value to a dictionary
        stem_grads_value_dict[country] = stem_grads_value

    except:
        print(country, 'not found')

In [ ]:
stem_grads_value_dict

In [ ]:
# I want to see all rows with country Bhtan
rows = df[df['primary_country'] == 'Trinidad and Tobago']
print(rows)

In [ ]:
# change all Afghanistan stem grads value to 11.24512004852295, remember to strip the space
df.loc[df['primary_country'].str.strip() == 'Bolivia', '% Stem Grads'] = 4.86275
# change change like this, but use a loop
for country in stem_grads_value_dict:
    # skip Italy and Japan
    if country in ['Italy', 'Japan']:
        continue
    df.loc[df['primary_country'].str.strip() == country, '% Stem Grads'] = stem_grads_value_dict[country]

In [ ]:
df

In [ ]:
# save to tsv
df.to_csv('df.tsv', index=False, sep='\t')